In [ ]:
import sys
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import math
import numpy as np

from scipy.stats import ttest_ind, pointbiserialr
from statsmodels.stats.multitest import multipletests

sys.path.append(os.path.abspath("src"))
from loaders import (
    DataCenter, DataSource, DataFile, SourceNames, CSVDataLoader,
    MOSAICWESDataCleaner, MOSAICClinicalDataCleaner
)

DATA_PATH = "/path/to/data_folder"

mosaic_data = DataCenter(
    name="MOSAIC",
    sources={
        SourceNames.wes: DataSource(
            files={
                "snv": DataFile(name=f"{DATA_PATH}/wes/snv_data.csv", loader=CSVDataLoader(data_transformer=MOSAICWESDataCleaner())),
                "cnv": DataFile(name=f"{DATA_PATH}/wes/cnv_data.csv", loader=CSVDataLoader(data_transformer=MOSAICWESDataCleaner()))
            }
        ),
        SourceNames.clinical: DataSource(
            files={"clinical": DataFile(name=f"{DATA_PATH}/clinical/metadata.xlsx", loader=CSVDataLoader(data_transformer=MOSAICClinicalDataCleaner()))}
        )
    }
)

In [ ]:
source_dict_mosaic = mosaic_data.load_tabular()

meta_path = '/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/mosaic_dataset/Clinical/GBM_HK_sample_and_clinical_data.csv'
clinical_cols: list[str] = ['os_years','pfs_years', 'sample_id', 'sample_collection_chronology']
id_col: str = 'sample_id'
meta = pd.read_csv(meta_path)
# normalize sample_id format (remove underscores)
meta[id_col] = meta[id_col].str.replace('_', '')
meta['sample_id_underscored'] = (
    meta['sample_id']
    .str.replace(r"^HKG(\d{3})([ab])$", r"HK_G_\1\2", regex=True)
)

### non-longitudinal WES analysis

In [ ]:
# Access SNVs and indels
snvs_indels = source_dict_mosaic["wes"]["WES mutations"]

baseline_ids = meta.loc[meta['sample_collection_chronology'] == "Baseline", 
                        'sample_id_underscored']
snvs_baseline = snvs_indels.loc[
    snvs_indels.index.isin(baseline_ids)
]
snvs_baseline

In [ ]:
# Access amplifications
CNVamp = source_dict_mosaic["wes"]["WES CNV amplification"]
CNVamp_baseline = CNVamp.loc[
    CNVamp.index.isin(baseline_ids)
]
CNVamp_baseline

In [ ]:
# Access deletions
CNVdel = source_dict_mosaic["wes"]["WES CNV deletion"]
CNVdel_baseline = CNVdel.loc[
    CNVdel.index.isin(baseline_ids)
]
CNVdel_baseline

In [ ]:
def plot_top_alterations_tcga_style(snv_df, amp_df, del_df, top_n=10,
                                   titles=("Most mutated genes (SNV)",
                                           "Most amplified genes (CNV)",
                                           "Most deleted genes (CNV)"),
                                   colors=("#009E73", "#D55E00", "#0072B2")):
    """
    Gene-centric barh panels styled like the 'Global abundance' (left) panel in your example:
    - x ticks removed
    - clean spines
    - constant annotation offset
    - xlim = max(max*1.15, floor)
    Assumes input dfs are boolean/int with rows=samples, cols=genes.
    """

    def get_top_data(df):
        counts = df.sum(axis=0)
        total_samples = df.shape[0]
        props = (counts / total_samples).fillna(0.0)
        top = props.sort_values(ascending=False).head(top_n)
        return top.sort_values(ascending=True)  # small bottom, large top

    snv_top = get_top_data(snv_df)
    amp_top = get_top_data(amp_df)
    del_top = get_top_data(del_df)

    fig_width_in = 250 / 25.4
    fig_height_in = max(3.8, 0.22 * top_n)  # match your left-panel heuristic

    fig, axes = plt.subplots(
        nrows=1, ncols=3,
        figsize=(fig_width_in, fig_height_in),
        dpi=300,
        gridspec_kw={"wspace": 0.35}  # slightly tighter like your example
    )

    for ax, data, title, color in zip(axes, [snv_top, amp_top, del_top], titles, colors):
        genes = data.index.tolist()
        vals = data.values.astype(float)

        ax.barh(
            range(len(genes)),
            vals,
            color=[color] * len(genes),   # list-of-colors like your left panel
            edgecolor="none",
            height=0.7
        )

        ax.set_yticks(range(len(genes)))
        ax.set_yticklabels(genes, fontsize=8)
        ax.set_xticks([])
        ax.set_xlabel(None)
        ax.set_title(title, fontsize=10, pad=8, loc="left")

        # annotate ALL percentages with constant offset (like +0.004)
        # use a small offset that scales with range, but keep the *style* constant
        maxv = float(np.max(vals)) if len(vals) else 0.0
        x_floor = 0.06  # same vibe as your left panel
        xlim_right = max(maxv * 1.15, x_floor)
        offset = 0.006 if xlim_right <= 1.0 else xlim_right * 0.01  # keep readable if ever >1

        for y, v in enumerate(vals):
            x_text = v + offset if v > 0 else offset
            ax.text(x_text, y, f"{v*100:.1f}%", va="center", ha="left", fontsize=8)

        # clean spines like your example
        for spine in ["top", "right", "bottom"]:
            ax.spines[spine].set_visible(False)

        # match y tick style: no tick marks
        ax.tick_params(axis="y", length=0)

        ax.set_xlim(0, xlim_right)
        ax.grid(False)

    plt.tight_layout()
    plt.savefig('genomic_alteration_frequencies.pdf')


In [ ]:
snv_gene_list = snvs_baseline.columns.tolist()

brennan_tcga_tableS2_SNV = [
     "TP53", "EGFR", "PTEN", "NF1", "PIK3CA", "PIK3R1",
     "SPTA1", "RB1", "TCHH", "ATRX", "KEL", "IDH1", "ABCC9",
     "PDGFRA", "STAG2", "COL1A2", "HEATR7B2", "NLRP5",
     "GABRA6", "SEMA3C", "SCN9A", "CDH18", "LZTR1",
     "ABCB1", "DRD5", "ADAM29", "SEMG1", "CXorf22",
     "TPTE2", "CDH9", "CALCR", "TRPV6", "DCAF12L2",
     "SEMA3E", "SPRYD5", "RPL5", "SULT1B1", "SLC26A3",
     "DYNC1I1", "ZNF99", "MMP13", "CARD6", "ZNF844", "LRRC55",
     "IL18RAP", "AFM", "BRAF", "UGT2A3", "GABRB2", "GABRA1",
     "FGA", "FRMD7", "SIGLEC8", "FOXR2", "ANKRD36", "WNT2",
     "LUM", "RFX6", "QKI", "GPX5", "PODNL1", "CD3EAP",
     "CDHR3", "C1orf150", "KRTAP20-2", "ODF4", "LCE4A",
     "CDX4", "PLCH2", "PARD6B", "TMEM147"
]

snvs_baseline_filtered = snvs_baseline.loc[:,snvs_baseline.columns.intersection(brennan_tcga_tableS2_SNV)]
snvs_baseline_filtered

In [ ]:
df = pd.read_csv("CNA_Genes.txt", sep="\t")

amp_genes = (
    df[(df["CNA"] == "AMP") &
       (df["Gistic(Q-value)"].notna()) &
       (df["Gistic(Q-value)"] <= 0.25) &
       (df["Is Cancer Gene (source: OncoKB)"] == "Yes")]
    ["Gene"]
    .unique()
)

del_genes = (
    df[(df["CNA"] == "HOMDEL") &
       (df["Gistic(Q-value)"].notna()) &
       (df["Gistic(Q-value)"] <= 0.25) &
       (df["Is Cancer Gene (source: OncoKB)"] == "Yes")]
    ["Gene"]
    .unique()
)

print(len(amp_genes))
print(len(del_genes))

In [ ]:
CNVamp_baseline_filtered = CNVamp_baseline.loc[:,CNVamp_baseline.columns.intersection(amp_genes)]

CNVdel_baseline_filtered = CNVdel_baseline.loc[:,CNVdel_baseline.columns.intersection(del_genes)]


In [ ]:
plot_top_alterations_tcga_style(
    snvs_baseline_filtered,
    CNVamp_baseline_filtered,
    CNVdel_baseline_filtered,
    top_n=10
)

### longitudinal WES analysis

In [ ]:
paired_ids = (
    meta.groupby("patient_id")["sample_collection_chronology"]
    .apply(lambda x: {"Baseline", "At recurrence"}.issubset(set(x)))
)

patients_with_pairs = paired_ids[paired_ids].index.tolist()

pairs = (
    meta[meta["patient_id"].isin(patients_with_pairs)]
    .groupby("patient_id")["sample_id_underscored"]
    .apply(list)
)



In [ ]:
snv_driver_genes = snvs_baseline_filtered.columns

snv_long = snvs_indels.loc[:, snvs_indels.columns.intersection(snv_driver_genes)]

CNVamp_long = CNVamp.loc[:, CNVamp.columns.intersection(amp_genes)]
CNVdel_long = CNVdel.loc[:, CNVdel.columns.intersection(del_genes)]

In [ ]:
def build_pair_results_from_bool(meta, pairs, df_bool):
    out = {}
    for patient in pairs.index:
        m = meta.loc[meta["patient_id"] == patient]
        b = m.query("sample_collection_chronology == 'Baseline'")
        r = m.query("sample_collection_chronology == 'At recurrence'")
        if len(b) != 1 or len(r) != 1:
            continue

        b_id = b.sample_id_underscored.iloc[0]
        r_id = r.sample_id_underscored.iloc[0]
        if (b_id not in df_bool.index) or (r_id not in df_bool.index):
            continue

        baseline = df_bool.loc[b_id].astype(bool)
        recur    = df_bool.loc[r_id].astype(bool)

        gained = recur[recur & ~baseline].index.tolist()
        lost   = baseline[baseline & ~recur].index.tolist()
        shared = baseline[baseline & recur].index.tolist()

        out[patient] = {"gained": gained, "lost": lost, "shared": shared}
    return out

def top_events_df(pair_results, top_n=2):
    gained = Counter()
    lost = Counter()
    for d in pair_results.values():
        gained.update(d["gained"])
        lost.update(d["lost"])

    events = sorted(set(gained) | set(lost))
    df = pd.DataFrame({
        "gained_n": [gained[e] for e in events],
        "lost_n":   [lost[e] for e in events],
    }, index=events)

    df["total_change"] = df["gained_n"] + df["lost_n"]
    df = df.sort_values(["total_change", "gained_n"], ascending=[False, False]).head(top_n)
    return df

def plot_top2_three_modalities_vertical_v4(
    meta, pairs, snv_bool, cnv_amp_bool, cnv_del_bool,
    top_n=2,
    titles=("SNV", "CNV amplification", "CNV deletion"),
    gain_color="#6A3D9A",   # purple
    loss_color="#B0B0B0",   # grey
    fig_width_mm=120,       # <- reduced from 180
    fig_height_in=3.0,
    savepath=None
):
    snv_pair = build_pair_results_from_bool(meta, pairs, snv_bool)
    amp_pair = build_pair_results_from_bool(meta, pairs, cnv_amp_bool)
    del_pair = build_pair_results_from_bool(meta, pairs, cnv_del_bool)

    df_snv = top_events_df(snv_pair, top_n=top_n)
    df_amp = top_events_df(amp_pair, top_n=top_n)
    df_del = top_events_df(del_pair, top_n=top_n)
    dfs = [df_snv, df_amp, df_del]

    # shared y-limits
    global_max = 1
    for df in dfs:
        if len(df):
            global_max = max(global_max, int(df[["gained_n", "lost_n"]].max().max()))
    pad = max(0.5, global_max * 0.25)

    fig_width_in = fig_width_mm / 25.4
    fig, axes = plt.subplots(
        1, 3,
        figsize=(fig_width_in, fig_height_in),
        dpi=300,
        gridspec_kw={"wspace": 0.45}  # slightly tighter
    )

    def draw_panel(ax, df, title):
        genes = df.index.tolist()
        x = np.arange(len(genes))

        gains = df["gained_n"].values.astype(float)
        losses = -df["lost_n"].values.astype(float)

        ax.bar(x, gains, width=0.55, color=gain_color, edgecolor="none")
        ax.bar(x, losses, width=0.55, color=loss_color, edgecolor="none")

        ax.axhline(0, linewidth=1, color="black")  # black 0-line

        ax.set_title(title, fontsize=11, pad=6, loc="center")
        ax.set_xticks(x)
        ax.set_xticklabels(genes, fontsize=9, rotation=35, ha="right")

        ax.set_yticks([])
        for spine in ["top", "right", "left"]:
            ax.spines[spine].set_visible(False)
        ax.grid(False)
        ax.set_ylim(-(global_max + pad), global_max + pad)

        # annotate counts
        for i, (g, lneg) in enumerate(zip(gains, losses)):
            l = -lneg
            if g > 0:
                ax.text(i, g + 0.1, f"{int(g)}", ha="center", va="bottom", fontsize=9)
            if l > 0:
                ax.text(i, -l - 0.1, f"{int(l)}", ha="center", va="top", fontsize=9)

    for ax, df, title in zip(axes, dfs, titles):
        draw_panel(ax, df, title)

    # place Gains/Losses between subplot 1 and 2
    pos0 = axes[0].get_position()
    pos1 = axes[1].get_position()
    x_mid = (pos0.x1 + pos1.x0) / 2.0
    y_mid = (pos0.y0 + pos0.y1) / 2.0
    fig.text(x_mid, y_mid + 0.03, "Gains", color=gain_color, fontsize=10,
             ha="center", va="center")
    fig.text(x_mid, y_mid - 0.03, "Losses", color=loss_color, fontsize=10,
             ha="center", va="center")

    # leave room for suptitle
    plt.tight_layout(rect=[0, 0, 1, 0.92])

    if savepath:
        plt.savefig(savepath, bbox_inches="tight")
    plt.show()

    return df_snv, df_amp, df_del

In [ ]:
df_snv, df_amp, df_del = plot_top2_three_modalities_vertical_v4(
    meta, pairs, snv_long, CNVamp_long, CNVdel_long,
    top_n=2,
    fig_width_mm=140,
    savepath="figures/longitudinal_WES_top2_three_panels.pdf"
)

### WES correlations with factor values

facs = pd.read_csv(f"/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/03_Factor_Data_v31_norecurr_bulkhvg_MOFA.csv")

In [ ]:
def normalize_sample_id(x: str) -> str:
    """
    Make sample IDs comparable between e.g.:
      'HK_G_001a'  -> 'HKG001a'
      'HKG001a'    -> 'HKG001a'
    Strategy: uppercase and drop all non-alphanumerics.
    """
    if pd.isna(x):
        return x
    x = str(x).upper()
    x = re.sub(r"[^A-Z0-9]", "", x)  # remove underscores, dashes, etc.
    return x


def prepare_wes_matrix(wes: pd.DataFrame, sample_id_col: str | None = None) -> pd.DataFrame:
    """
    Accepts WES matrices in either form:
      - index is sample_id
      - or sample_id is a column (provide sample_id_col)

    Returns a dataframe indexed by normalized sample_id with values in {0,1} or NaN.
    """
    w = wes.copy()

    if sample_id_col is not None:
        w[sample_id_col] = w[sample_id_col].map(normalize_sample_id)
        w = w.set_index(sample_id_col)
    else:
        w.index = w.index.map(normalize_sample_id)

    # Convert bool -> int, keep NaN as NaN
    w = w.applymap(lambda v: np.nan if pd.isna(v) else int(bool(v)))

    # Ensure columns are strings (gene names)
    w.columns = w.columns.astype(str)

    return w


def prepare_factors(facs: pd.DataFrame, sample_id_col: str = "sample_id") -> tuple[pd.DataFrame, list[str]]:
    f = facs.copy()
    f[sample_id_col] = f[sample_id_col].map(normalize_sample_id)

    # Auto-detect factor columns: Factor<number>
    factor_cols = [c for c in f.columns if re.fullmatch(r"Factor\d+", str(c))]
    factor_cols = sorted(factor_cols, key=lambda x: int(re.findall(r"\d+", x)[0]))

    return f[[sample_id_col] + factor_cols], factor_cols

def factor_wes_association(
    facs: pd.DataFrame,
    wes: pd.DataFrame,
    wes_label: str,
    sample_id_col: str = "sample_id",
    min_n_per_group: int = 4,
    alpha: float = 0.05,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    For each gene and each factor:
      - point-biserial correlation (r, p)
      - Welch t-test (t, p)
      - BH/FDR on the chosen p-values (here: correlation p-value)

    Returns:
      res_df: all tests
      sig_df: significant tests by q < alpha
    """
    facs_prep, factor_cols = prepare_factors(facs, sample_id_col=sample_id_col)
    wes_prep = prepare_wes_matrix(wes, sample_id_col=None)

    # Merge on normalized IDs
    merged = facs_prep.merge(
        wes_prep.reset_index().rename(columns={"index": sample_id_col}),
        on=sample_id_col,
        how="inner",
    )

    # Genes = wes columns
    gene_cols = [c for c in wes_prep.columns if c in merged.columns]

    results = []

    for gene in gene_cols:
        status = merged[gene]  # 0/1/NaN

        for fac in factor_cols:
            x = merged[fac]
            mask = status.notna() & x.notna()
            if mask.sum() == 0:
                continue

            s = status[mask].astype(int)
            xf = x[mask].astype(float)

            n1 = int((s == 1).sum())
            n0 = int((s == 0).sum())

            # Require enough samples in both groups
            if n1 < min_n_per_group or n0 < min_n_per_group:
                results.append({
                    "wes": wes_label,
                    "gene": gene,
                    "factor": fac,
                    "n0": n0,
                    "n1": n1,
                    "r_pb": np.nan,
                    "p_pb": np.nan,
                    "t_welch": np.nan,
                    "p_t": np.nan,
                    "mean0": float(xf[s == 0].mean()) if n0 > 0 else np.nan,
                    "mean1": float(xf[s == 1].mean()) if n1 > 0 else np.nan,
                    "delta_mean1_minus_mean0": (float(xf[s == 1].mean()) - float(xf[s == 0].mean())) if (n0 > 0 and n1 > 0) else np.nan,
                })
                continue

            # point-biserial correlation (equivalent to Pearson with a 0/1 variable)
            r_pb, p_pb = pointbiserialr(s, xf)

            # Welch t-test
            grp1 = xf[s == 1]
            grp0 = xf[s == 0]
            t_stat, p_t = ttest_ind(grp1, grp0, equal_var=False, nan_policy="omit")

            results.append({
                "wes": wes_label,
                "gene": gene,
                "factor": fac,
                "n0": n0,
                "n1": n1,
                "r_pb": r_pb,
                "p_pb": p_pb,
                "t_welch": t_stat,
                "p_t": p_t,
                "mean0": float(grp0.mean()),
                "mean1": float(grp1.mean()),
                "delta_mean1_minus_mean0": float(grp1.mean() - grp0.mean()),
            })

    res_df = pd.DataFrame(results)

    # BH/FDR on Welch t-test p-values (group difference)
    mask = res_df["p_t"].notna()
    if mask.any():
        rej, qvals, _, _ = multipletests(res_df.loc[mask, "p_t"], alpha=alpha, method="fdr_bh")
        res_df.loc[mask, "q_t"] = qvals
        res_df.loc[mask, "rej_t"] = rej
    else:
        res_df["q_t"] = np.nan
        res_df["rej_t"] = False
    
    sig_df = res_df[(res_df["q_t"] < alpha)].sort_values("q_t")

    return res_df, sig_df

def plot_significant_boxplots(
    facs: pd.DataFrame,
    wes: pd.DataFrame,
    sig_df: pd.DataFrame,
    wes_label: str,
    sample_id_col: str = "sample_id",
    max_plots: int = 12,
    ncols: int = 3,
    panel_w: float = 2,
    panel_h: float = 2.1,
    palette=("0.85", "0.55"),
    savepath: str | None = None,
):
    if sig_df.empty:
        print(f"No significant results for {wes_label}.")
        return

    sig_df = sig_df.head(max_plots).copy()

    facs_prep, _ = prepare_factors(facs, sample_id_col=sample_id_col)
    wes_prep = prepare_wes_matrix(wes, sample_id_col=None)

    merged = facs_prep.merge(
        wes_prep.reset_index().rename(columns={"index": sample_id_col}),
        on=sample_id_col,
        how="inner",
    )

    n = len(sig_df)
    ncols = min(ncols, n)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols * panel_w, nrows * panel_h),
        constrained_layout=True
    )
    axes = np.atleast_1d(axes).ravel()

    for i, (_, row) in enumerate(sig_df.iterrows()):
        ax = axes[i]
        gene = row["gene"]
        fac = row["factor"]

        dfp = merged[[gene, fac]].dropna().copy()
        dfp[gene] = dfp[gene].astype(int)

        n0 = int((dfp[gene] == 0).sum())
        n1 = int((dfp[gene] == 1).sum())

        sns.boxplot(
            x=gene, y=fac, data=dfp, ax=ax,
            width=0.5, showfliers=False,
            palette=palette,
            linewidth=0.9,
            medianprops=dict(linewidth=1.0),
            whiskerprops=dict(linewidth=0.9),
            capprops=dict(linewidth=0.9),
        )

        sns.stripplot(
            x=gene, y=fac, data=dfp, ax=ax,
            jitter=0.18, size=2.5,
            alpha=0.55, linewidth=0,
            color="0.25"
        )

        # short title (no stats here)
        ax.set_title(f"{gene} – {fac}", pad=2)

        # x tick labels with n
        ax.set_xlabel("")
        ax.set_xticklabels([f"0 (n={n0})", f"1 (n={n1})"])

        # y label only on left column (clean multipanel look)
        if (i % ncols) == 0:
            ax.set_ylabel("Factor value")
        else:
            ax.set_ylabel("")

        # in-axes stats annotation (small, subtle)
        q = row["q_t"]
        t = row["t_welch"]
        dm = row.get("delta_mean1_minus_mean0", np.nan)
        stat_txt = f"q={q:.2g} | t={t:.2f} | Δ={dm:.2f}"
        #ax.text(
        #    0.02, 0.98, stat_txt,
        #    transform=ax.transAxes,
        #    ha="left", va="top",
        #    fontsize=7,
        #    color="0.25"
        #)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # hide unused axes
    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    if savepath is not None:
        fig.savefig(savepath)
    plt.show()

In [ ]:
alpha = 0.05
min_n = 3

# SNV
snv_res, snv_sig = factor_wes_association(
    facs=facs,
    wes=snvs_baseline_filtered,
    wes_label="SNV",
    min_n_per_group=min_n,
    alpha=alpha,
)

# CNV amp
amp_res, amp_sig = factor_wes_association(
    facs=facs,
    wes=CNVamp_baseline_filtered,
    wes_label="CNV_AMP",
    min_n_per_group=min_n,
    alpha=alpha,
)

# CNV del
del_res, del_sig = factor_wes_association(
    facs=facs,
    wes=CNVdel_baseline_filtered,
    wes_label="CNV_DEL",
    min_n_per_group=min_n,
    alpha=alpha,
)


In [ ]:
plot_significant_boxplots(facs, snvs_baseline_filtered, snv_sig, wes_label="SNV")
plot_significant_boxplots(facs, CNVamp_baseline_filtered, amp_sig, wes_label="CNV_AMP")
plot_significant_boxplots(facs, CNVdel_baseline_filtered, del_sig, wes_label="CNV_DEL")